In [ ]:
from pathlib import Path
import os
import json
import pandas as pd

# Locate the project root.
current = Path.cwd().resolve()

project_root = next(
    (
        folder
        for folder in [current, *current.parents]
        if (folder / ".git").exists()
    ),
    None
)

if project_root is None:
    raise RuntimeError(
        "Please open this notebook inside the Posters repository."
    )

os.chdir(project_root)

raw_folder = Path("data/raw/posters_learning")

records = json.loads(
    (raw_folder / "metadata_full.json").read_text(
        encoding="utf-8"
    )
)

posters = pd.DataFrame(records)

assert posters["poster_id"].is_unique

print("Poster records loaded:", len(posters))

In [ ]:
date_frequency = (
    posters["date_raw"]
    .value_counts(dropna=False)
    .rename_axis("date_raw")
    .reset_index(name="record_count")
)

display(date_frequency.head(30))

In [ ]:
import re

month_names = (
    "January|February|March|April|May|June|"
    "July|August|September|October|November|December"
)

exact_year_pattern = re.compile(
    r"^(?:18|19|20)\d{2}$"
)

exact_month_pattern = re.compile(
    rf"^(?P<year>(?:18|19|20)\d{{2}})"
    rf"\s*,?\s+(?P<month>{month_names})$",
    flags=re.IGNORECASE
)


def parse_poster_date(value):
    """Accept unambiguous year or year-month strings only."""
    raw = "" if pd.isna(value) else str(value).strip()
    normalized = re.sub(r"\s+", " ", raw)

    if exact_year_pattern.fullmatch(normalized):
        return {
            "poster_year": int(normalized),
            "date_precision": "year",
            "date_review_required": False
        }

    month_match = exact_month_pattern.fullmatch(normalized)

    if month_match:
        return {
            "poster_year": int(month_match.group("year")),
            "date_precision": "month",
            "date_review_required": False
        }

    return {
        "poster_year": pd.NA,
        "date_precision": "unresolved",
        "date_review_required": True
    }


date_features = pd.DataFrame(
    posters["date_raw"].apply(parse_poster_date).tolist(),
    index=posters.index
)

date_features["poster_year"] = (
    date_features["poster_year"].astype("Int64")
)

# Replace derived columns safely if this cell is rerun.
posters = posters.drop(
    columns=date_features.columns,
    errors="ignore"
).join(date_features)

print("Date parsing status:")
print(posters["date_precision"].value_counts())

In [ ]:
posters["eligible_year_for_next_year_y"] = (
    posters["poster_year"]
    .between(1949, 2013)
    .fillna(False)
)

print(
    "Posters with an eligible, unambiguous year:",
    int(posters["eligible_year_for_next_year_y"].sum())
)

eligible_year_counts = (
    posters.loc[
        posters["eligible_year_for_next_year_y"]
    ]
    .groupby("poster_year")
    .size()
    .rename("poster_count")
)

# Include candidate years with no retained posters.
eligible_year_counts = eligible_year_counts.reindex(
    range(1949, 2014),
    fill_value=0
)

print(
    "Years with at least one eligible poster:",
    int((eligible_year_counts > 0).sum())
)

display(eligible_year_counts.to_frame())

In [ ]:
review_folder = Path("data/review")
processed_folder = Path("data/processed")

review_folder.mkdir(parents=True, exist_ok=True)
processed_folder.mkdir(parents=True, exist_ok=True)

date_review = posters.loc[
    posters["date_review_required"],
    ["poster_id", "source_url", "date_raw"]
].copy()

date_review["reviewed_year"] = ""
date_review["review_evidence"] = ""

# Do not overwrite later manual annotations.
review_path = review_folder / "poster_dates_to_review.csv"

if not review_path.exists():
    date_review.to_csv(
        review_path,
        index=False,
        encoding="utf-8-sig"
    )
else:
    print("Existing date review file preserved.")

posters[
    [
        "poster_id",
        "date_raw",
        "poster_year",
        "date_precision",
        "date_review_required",
        "eligible_year_for_next_year_y"
    ]
].to_csv(
    processed_folder / "poster_date_features.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Date features saved.")

In [ ]:
%pip install numpy scikit-image Pillow

In [ ]:
import io
import numpy as np

from PIL import Image, ImageOps, ImageCms
from skimage.color import rgb2hsv, rgb2lab


def extract_color_features(image_path, return_preview=False):
    """Calculate full-image color features in sRGB."""
    with Image.open(image_path) as source:
        icc_profile = source.info.get("icc_profile")

        image = ImageOps.exif_transpose(source)

        # Reject transparency instead of choosing an arbitrary background.
        if image.mode == "P":
            image = image.convert("RGBA")

        if "A" in image.getbands():
            if image.getchannel("A").getextrema()[0] < 255:
                raise ValueError(
                    "Transparent image requires a defined background policy."
                )

            image = image.convert("RGB")

        if icc_profile:
            input_profile = ImageCms.ImageCmsProfile(
                io.BytesIO(icc_profile)
            )

            image = ImageCms.profileToProfile(
                image,
                input_profile,
                ImageCms.createProfile("sRGB"),
                outputMode="RGB",
                renderingIntent=1
            )

            profile_status = "icc_converted_to_srgb"

        elif image.mode in ["RGB", "L", "1"]:
            image = image.convert("RGB")
            profile_status = "assumed_srgb"

        else:
            raise ValueError(
                f"Unsupported image mode without ICC: {image.mode}"
            )

        rgb = np.asarray(image, dtype=np.float64) / 255.0

    hsv = rgb2hsv(rgb)

    hue = hsv[..., 0]
    saturation = hsv[..., 1]
    value = hsv[..., 2]

    red_mask = (
        ((hue <= 15 / 360) | (hue >= 345 / 360))
        & (saturation >= 0.40)
        & (value >= 0.15)
    )

    lab = rgb2lab(
        rgb,
        illuminant="D65",
        observer="2"
    )

    lightness = lab[..., 0]

    features = {
        "red_ratio": float(red_mask.mean()),
        "r_mean": float(rgb[..., 0].mean()),
        "g_mean": float(rgb[..., 1].mean()),
        "b_mean": float(rgb[..., 2].mean()),
        "saturation_mean": float(saturation.mean()),
        "lightness_mean": float(lightness.mean()),
        "lightness_std": float(lightness.std(ddof=0)),
        "lightness_p90_p10": float(
            np.percentile(lightness, 90)
            - np.percentile(lightness, 10)
        ),
        "color_profile_status": profile_status
    }

    if return_preview:
        return features, rgb, red_mask

    return features


print("Color extraction function is ready.")

In [ ]:
eligible_posters = posters.loc[
    posters["eligible_year_for_next_year_y"]
].copy()

test_posters = eligible_posters.head(5)

test_color_rows = []

for _, row in test_posters.iterrows():
    # Use the first display image as the primary image.
    image_info = row["images"][0]
    image_path = raw_folder / image_info["image_path"]

    features = extract_color_features(image_path)

    test_color_rows.append({
        "poster_id": row["poster_id"],
        "poster_year": row["poster_year"],
        **features
    })

test_colors = pd.DataFrame(test_color_rows)

assert test_colors["red_ratio"].between(0, 1).all()
assert test_colors[
    ["r_mean", "g_mean", "b_mean", "saturation_mean"]
].apply(lambda column: column.between(0, 1).all()).all()

display(test_colors)

In [ ]:
preview_candidates = eligible_posters.copy()

preview_candidates["decade"] = (
    preview_candidates["poster_year"] // 10 * 10
)

# Select one poster per available decade for an initial technical check.
preview_sample = (
    preview_candidates
    .sort_values(["poster_year", "poster_id"])
    .groupby("decade", sort=True)
    .head(1)
)

display(
    preview_sample[
        ["poster_id", "poster_year", "decade"]
    ]
)

In [ ]:
%pip install matplotlib

In [ ]:
import matplotlib.pyplot as plt

for _, row in preview_sample.iterrows():
    image_info = row["images"][0]
    image_path = raw_folder / image_info["image_path"]

    features, rgb, red_mask = extract_color_features(
        image_path,
        return_preview=True
    )

    # Mark pixels classified as red using a cyan overlay.
    overlay = rgb.copy()
    cyan = np.array([0.0, 1.0, 1.0])

    overlay[red_mask] = (
        0.5 * rgb[red_mask]
        + 0.5 * cyan
    )

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    axes[0].imshow(rgb)
    axes[0].set_title("Original")

    axes[1].imshow(
        red_mask,
        cmap="gray",
        vmin=0,
        vmax=1
    )
    axes[1].set_title(
        f"Red mask: {features['red_ratio']:.1%}"
    )

    axes[2].imshow(overlay)
    axes[2].set_title("Detected red pixels in cyan")

    for axis in axes:
        axis.axis("off")

    fig.suptitle(
        f"{row['poster_id']} | {int(row['poster_year'])}"
    )

    plt.tight_layout()
    plt.show()

In [ ]:
import inspect
import hashlib
import json
import time
from datetime import datetime, timezone

color_folder = Path("data/processed/color_v1")
checkpoint_folder = color_folder / "records"

checkpoint_folder.mkdir(parents=True, exist_ok=True)

# Include the function definition so modified code invalidates old checkpoints.
function_source = inspect.getsource(extract_color_features)

color_config = {
    "version": "color_v1",
    "hue_half_width_degrees": 15,
    "saturation_min": 0.40,
    "value_min": 0.15,
    "region": "full_image",
    "image_selection": "first_display_image",
    "lightness": "CIELAB L*, D65, 2-degree observer",
    "function_sha256": hashlib.sha256(
        function_source.encode("utf-8")
    ).hexdigest()
}

config_sha256 = hashlib.sha256(
    json.dumps(color_config, sort_keys=True).encode("utf-8")
).hexdigest()

(color_folder / "config.json").write_text(
    json.dumps(color_config, indent=2),
    encoding="utf-8"
)

print("Color configuration saved.")

In [ ]:
color_rows = []
started_at = time.monotonic()

for index, (_, row) in enumerate(
    eligible_posters.iterrows(),
    start=1
):
    poster_id = row["poster_id"]
    checkpoint_path = checkpoint_folder / f"{poster_id}.json"

    result = None

    # Reuse successful checkpoints only when the configuration
    # and input image checksum match.
    image_info = row["images"][0]
    image_sha256 = image_info["image_sha256"]

    if checkpoint_path.exists():
        saved = json.loads(
            checkpoint_path.read_text(encoding="utf-8")
        )

        if (
            saved.get("color_status") == "ok"
            and saved.get("config_sha256") == config_sha256
            and saved.get("image_sha256") == image_sha256
        ):
            result = saved

    if result is None:
        result = {
            "poster_id": poster_id,
            "image_sha256": image_sha256,
            "config_sha256": config_sha256
        }

        try:
            image_path = raw_folder / image_info["image_path"]

            result.update(
                extract_color_features(image_path)
            )

            result["color_status"] = "ok"
            result["color_error"] = ""

        except Exception as error:
            result["color_status"] = "error"
            result["color_error"] = str(error)

        result["processed_at_utc"] = datetime.now(
            timezone.utc
        ).isoformat()

        temporary_path = checkpoint_path.with_suffix(".json.tmp")

        temporary_path.write_text(
            json.dumps(result, indent=2),
            encoding="utf-8"
        )

        temporary_path.replace(checkpoint_path)

    color_rows.append(result)

    if index % 100 == 0 or index == len(eligible_posters):
        failure_count = sum(
            item["color_status"] == "error"
            for item in color_rows
        )

        elapsed = time.monotonic() - started_at

        print(
            f"Processed {index}/{len(eligible_posters)} | "
            f"Errors: {failure_count} | "
            f"Elapsed: {elapsed:.1f} seconds",
            flush=True
        )

color_features = pd.DataFrame(color_rows)

print("Color extraction finished.")

In [ ]:
assert color_features["poster_id"].is_unique
assert len(color_features) == len(eligible_posters)

successful_colors = color_features.loc[
    color_features["color_status"] == "ok"
]

assert successful_colors["red_ratio"].between(0, 1).all()

color_features.to_csv(
    color_folder / "poster_color_features.csv",
    index=False,
    encoding="utf-8-sig"
)

color_features.loc[
    color_features["color_status"] == "error"
].to_csv(
    color_folder / "color_errors.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Color extraction status:")
print(color_features["color_status"].value_counts())

display(
    successful_colors[
        ["red_ratio", "lightness_mean", "saturation_mean"]
    ].describe()
)

In [ ]:
display(
    eligible_posters["size_raw"]
    .value_counts(dropna=False)
    .head(20)
)

In [ ]:
import re
import unicodedata

size_pattern = re.compile(
    r"^\s*"
    r"(\d+(?:\.\d+)?)"
    r"\s*[xX×]\s*"
    r"(\d+(?:\.\d+)?)"
    r"\s*cm\.?\s*$",
    flags=re.IGNORECASE
)


def parse_physical_size(value):
    """Parse unambiguous two-dimensional sizes in centimeters."""
    raw = "" if pd.isna(value) else str(value)

    normalized = unicodedata.normalize("NFKC", raw).strip()

    result = {
        "size_short_cm": float("nan"),
        "size_long_cm": float("nan"),
        "size_area_cm2": float("nan"),
        "size_aspect_ratio": float("nan"),
        "size_parse_status": "missing" if not normalized else "review"
    }

    match = size_pattern.fullmatch(normalized)

    if match is None:
        return result

    side_1, side_2 = map(float, match.groups())

    if side_1 <= 0 or side_2 <= 0:
        result["size_parse_status"] = "invalid"
        return result

    short_side, long_side = sorted([side_1, side_2])

    result.update({
        "size_short_cm": short_side,
        "size_long_cm": long_side,
        "size_area_cm2": short_side * long_side,
        "size_aspect_ratio": long_side / short_side,
        "size_parse_status": "parsed"
    })

    return result

In [ ]:
size_values = pd.DataFrame(
    [
        parse_physical_size(value)
        for value in eligible_posters["size_raw"]
    ]
)

size_features = pd.concat(
    [
        eligible_posters[
            ["poster_id", "size_raw"]
        ].reset_index(drop=True),
        size_values
    ],
    axis=1
)

assert size_features["poster_id"].is_unique

print("Physical size parsing status:")
print(size_features["size_parse_status"].value_counts())

parsed_sizes = size_features.loc[
    size_features["size_parse_status"] == "parsed"
]

assert (
    parsed_sizes["size_long_cm"]
    >= parsed_sizes["size_short_cm"]
).all()

display(
    parsed_sizes[
        [
            "size_short_cm",
            "size_long_cm",
            "size_area_cm2",
            "size_aspect_ratio"
        ]
    ].describe()
)

In [ ]:
processed_folder = Path("data/processed")
review_folder = Path("data/review")

processed_folder.mkdir(parents=True, exist_ok=True)
review_folder.mkdir(parents=True, exist_ok=True)

size_features.to_csv(
    processed_folder / "poster_size_features.csv",
    index=False,
    encoding="utf-8-sig"
)

size_review = size_features.loc[
    size_features["size_parse_status"] != "parsed"
].copy()

size_review.to_csv(
    review_folder / "poster_sizes_review_candidates.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    size_review[
        ["size_raw", "size_parse_status"]
    ].value_counts(dropna=False).head(20)
)

In [ ]:
visual_features = (
    eligible_posters[
        ["poster_id", "source_url", "date_raw", "poster_year"]
    ]
    .merge(
        color_features,
        on="poster_id",
        how="left",
        validate="one_to_one",
        indicator="color_join"
    )
    .merge(
        size_features,
        on="poster_id",
        how="left",
        validate="one_to_one",
        indicator="size_join"
    )
)

assert len(visual_features) == len(eligible_posters)
assert visual_features["color_join"].eq("both").all()
assert visual_features["size_join"].eq("both").all()
assert visual_features["color_status"].eq("ok").all()

visual_features = visual_features.drop(
    columns=["color_join", "size_join"]
)

visual_features.to_csv(
    processed_folder / "poster_visual_features_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Visual feature rows:", len(visual_features))
print("Visual feature columns:", len(visual_features.columns))

In [ ]:
# Inspect unsupported or missing size formats.
display(
    size_features.loc[
        size_features["size_parse_status"] != "parsed",
        ["poster_id", "size_raw", "size_parse_status"]
    ]
)

# Inspect the largest parsed sizes; do not remove them automatically.
display(
    visual_features.nlargest(10, "size_area_cm2")[
        [
            "poster_id",
            "source_url",
            "size_raw",
            "size_long_cm",
            "size_area_cm2"
        ]
    ]
)

In [ ]:
assert visual_features["image_sha256"].notna().all()
assert visual_features["image_sha256"].ne("").all()

exact_duplicate_mask = visual_features.duplicated(
    subset=["image_sha256"],
    keep=False
)

exact_duplicate_records = (
    visual_features.loc[exact_duplicate_mask]
    .sort_values(["image_sha256", "poster_year", "poster_id"])
)

print("Records in exact-duplicate groups:",
      len(exact_duplicate_records))

print("Exact-duplicate groups:",
      exact_duplicate_records["image_sha256"].nunique())

display(
    exact_duplicate_records[
        [
            "poster_id",
            "poster_year",
            "image_sha256",
            "source_url"
        ]
    ].head(20)
)

In [ ]:
review_folder = Path("data/review")
review_folder.mkdir(parents=True, exist_ok=True)

exact_duplicate_records.to_csv(
    review_folder / "exact_image_duplicate_candidates.csv",
    index=False,
    encoding="utf-8-sig"
)

visual_features.nlargest(10, "size_area_cm2").to_csv(
    review_folder / "largest_size_candidates.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Review candidates saved.")

In [ ]:
import unicodedata
import re


def normalize_size_text(value):
    """Normalize explicit formatting issues without inferring units."""
    raw = "" if pd.isna(value) else str(value)

    text = unicodedata.normalize("NFKC", raw)
    text = text.replace("\ufeff", "").strip()

    changes = []

    if text != raw:
        changes.append("unicode_or_whitespace_normalization")

    # Convert decimal commas only when located between digits.
    updated = re.sub(r"(?<=\d),(?=\d)", ".", text)
    if updated != text:
        changes.append("decimal_comma")
        text = updated

    # Explicit observed formatting corrections.
    corrections = {
        "52.5x76 xcm.": "52.5x76 cm.",
        "53.5.x77 cm.": "53.5x77 cm.",
        "53xx77 cm.": "53x77 cm.",
        "53.x76.5 cm.": "53x76.5 cm.",
        "53x77 cmm.": "53x77 cm."
    }

    if text in corrections:
        text = corrections[text]
        changes.append("explicit_format_correction")

    return {
        "size_normalized": text,
        "size_normalization_rule": (
            "; ".join(changes) if changes else "unchanged"
        )
    }

In [ ]:
normalization = pd.DataFrame([
    normalize_size_text(value)
    for value in eligible_posters["size_raw"]
])

parsed_values = pd.DataFrame([
    parse_physical_size(value)
    for value in normalization["size_normalized"]
])

size_features_v2 = pd.concat(
    [
        eligible_posters[
            ["poster_id", "size_raw"]
        ].reset_index(drop=True),
        normalization,
        parsed_values
    ],
    axis=1
)

assert size_features_v2["poster_id"].is_unique
assert len(size_features_v2) == len(eligible_posters)

print("Updated size parsing status:")
print(size_features_v2["size_parse_status"].value_counts())

display(
    size_features_v2.loc[
        size_features_v2["size_parse_status"] != "parsed",
        ["poster_id", "size_raw", "size_parse_status"]
    ]
)

size_features_v2.to_csv(
    processed_folder / "poster_size_features_v2.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
visual_features = (
    eligible_posters[
        ["poster_id", "source_url", "date_raw", "poster_year"]
    ]
    .merge(
        color_features,
        on="poster_id",
        how="left",
        validate="one_to_one",
        indicator="color_join"
    )
    .merge(
        size_features_v2,
        on="poster_id",
        how="left",
        validate="one_to_one",
        indicator="size_join"
    )
)

assert visual_features["color_join"].eq("both").all()
assert visual_features["size_join"].eq("both").all()
assert len(visual_features) == len(eligible_posters)

visual_features = visual_features.drop(
    columns=["color_join", "size_join"]
)

visual_features.to_csv(
    processed_folder / "poster_visual_features_v2.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Updated visual features saved.")

In [ ]:
duplicate_mask = visual_features.duplicated(
    "image_sha256",
    keep=False
)

print(
    "Records in exact-duplicate groups:",
    int(duplicate_mask.sum())
)

print(
    "Exact-duplicate groups:",
    visual_features.loc[
        duplicate_mask, "image_sha256"
    ].nunique()
)

In [ ]:
visual_columns = [
    "red_ratio",
    "r_mean",
    "g_mean",
    "b_mean",
    "saturation_mean",
    "lightness_mean",
    "lightness_std",
    "lightness_p90_p10",
    "size_short_cm",
    "size_long_cm",
    "size_area_cm2",
    "size_aspect_ratio"
]

assert visual_features["poster_id"].is_unique
assert visual_features["poster_year"].notna().all()

assert all(
    column in visual_features.columns
    for column in visual_columns
), "Some visual feature columns are missing."

In [ ]:
annual_rows = []

for year, group in visual_features.groupby("poster_year"):
    annual_row = {
        "poster_year": int(year),
        "n_posters": len(group)
    }

    for column in visual_columns:
        values = group[column]

        annual_row[f"{column}_mean"] = values.mean()
        annual_row[f"{column}_median"] = values.median()
        annual_row[f"{column}_n_valid"] = int(values.notna().sum())
        annual_row[f"{column}_missing_ratio"] = float(
            values.isna().mean()
        )

    annual_rows.append(annual_row)

annual_visual_x = (
    pd.DataFrame(annual_rows)
    .sort_values("poster_year")
    .reset_index(drop=True)
)

assert annual_visual_x["poster_year"].is_unique
assert annual_visual_x["n_posters"].sum() == len(visual_features)

print("Annual X observations:", len(annual_visual_x))

display(
    annual_visual_x[
        [
            "poster_year",
            "n_posters",
            "red_ratio_mean",
            "lightness_mean_mean",
            "size_area_cm2_mean",
            "size_area_cm2_n_valid"
        ]
    ].head(15)
)

In [ ]:
annual_visual_x.to_csv(
    processed_folder / "annual_visual_x_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Annual visual X saved.")

In [ ]:
target_path = Path(
    "data/processed/military/annual_military_target.csv"
)

assert target_path.exists(), "The military target file was not found."

military_target = pd.read_csv(target_path)

assert military_target["year"].is_unique
assert military_target["y_active_mid_count"].notna().all()

# Poster year t predicts dispute count in year t+1.
annual_visual_x["target_year"] = (
    annual_visual_x["poster_year"] + 1
)

annual_model_table = annual_visual_x.merge(
    military_target.rename(columns={"year": "target_year"}),
    on="target_year",
    how="left",
    validate="one_to_one",
    indicator="target_join"
)

print("Target matching status:")
print(annual_model_table["target_join"].value_counts())

assert annual_model_table["target_join"].eq("both").all()

annual_model_table = annual_model_table.drop(
    columns="target_join"
)

annual_model_table.to_csv(
    processed_folder / "annual_visual_model_table_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    annual_model_table[
        [
            "poster_year",
            "target_year",
            "n_posters",
            "red_ratio_mean",
            "y_active_mid_count"
        ]
    ].head(10)
)

In [ ]:
import re
import unicodedata
import numpy as np

# Source: Wong (2026), Table 1.
THREAT_LEXICON = [
    "严重后果",
    "一切后果",
    "后果",
    "恶果",
    "消极影响",
    "负面影响",
    "不良后果",
    "适得其反",
    "恶劣影响",
    "后果严重",
    "自食恶果",
    "自食其果",
    "不顾后果",
    "玩火自焚",
    "自取灭亡",
    "引火自焚",
    "自寻死路",
    "苦果",
    "玩火"
]

# Match longer expressions first to avoid nested double counting.
THREAT_PATTERN = re.compile(
    "|".join(
        re.escape(term)
        for term in sorted(
            THREAT_LEXICON,
            key=lambda term: (-len(term), term)
        )
    )
)


def extract_title_dictionary_features(value):
    """Match fixed expressions in available Chinese title text."""
    raw = "" if pd.isna(value) else str(value)

    normalized = unicodedata.normalize("NFKC", raw).strip()

    han_count = len(
        re.findall(r"[\u4e00-\u9fff]", normalized)
    )

    # Remove whitespace for phrase matching.
    match_text = re.sub(r"\s+", "", normalized)

    matches = THREAT_PATTERN.findall(match_text)

    has_chinese = han_count > 0

    return {
        "title_zh_normalized": normalized,
        "title_has_chinese": int(has_chinese),
        "title_han_count": han_count,
        "title_threat_any": (
            int(bool(matches)) if has_chinese else np.nan
        ),
        "title_threat_count": (
            len(matches) if has_chinese else np.nan
        ),
        "title_threat_per100han": (
            100 * len(matches) / han_count
            if has_chinese
            else np.nan
        ),
        "title_threat_matches": matches
    }

In [ ]:
test_result = extract_title_dictionary_features(
    "严重后果，一切后果"
)

print(test_result)

assert test_result["title_threat_count"] == 2
assert test_result["title_threat_matches"] == [
    "严重后果",
    "一切后果"
]

missing_result = extract_title_dictionary_features("")

assert pd.isna(missing_result["title_threat_count"])

print("Dictionary matching checks passed.")

In [ ]:
title_values = pd.DataFrame([
    extract_title_dictionary_features(value)
    for value in eligible_posters["title_zh"]
])

title_features = pd.concat(
    [
        eligible_posters[
            ["poster_id", "poster_year", "title_zh"]
        ].reset_index(drop=True),
        title_values
    ],
    axis=1
)

assert title_features["poster_id"].is_unique
assert len(title_features) == len(eligible_posters)

available_titles = title_features.loc[
    title_features["title_has_chinese"] == 1
]

matched_titles = title_features.loc[
    title_features["title_threat_any"] == 1
].copy()

print("Eligible posters:", len(title_features))
print("Titles containing Chinese:", len(available_titles))
print("Titles with dictionary matches:", len(matched_titles))

if len(available_titles) > 0:
    print(
        "Match rate among available Chinese titles:",
        f"{len(matched_titles) / len(available_titles):.2%}"
    )

display(
    matched_titles[
        [
            "poster_id",
            "poster_year",
            "title_zh",
            "title_threat_matches",
            "title_threat_count"
        ]
    ]
)

In [ ]:
# Retain descriptive title features; exclude unsupported threat scores.
title_features_selected = title_features[
    [
        "poster_id",
        "title_has_chinese",
        "title_han_count"
    ]
].copy()

title_features_selected.to_csv(
    processed_folder / "poster_title_features_selected.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Threat dictionary features excluded from the model candidates.")

In [ ]:
dictionary_check = {
    "eligible_posters": len(title_features),
    "titles_with_chinese": len(available_titles),
    "matched_titles": len(matched_titles),
    "decision": "Exclude dictionary-derived threat features",
    "reason": (
        "No matches in available Chinese titles; "
        "the dictionary provides no observed threat-score variation."
    ),
    "scope": "Website Chinese titles, not complete poster text"
}

(processed_folder / "title_dictionary_check.json").write_text(
    json.dumps(dictionary_check, indent=2),
    encoding="utf-8"
)

print("Dictionary coverage check saved.")

In [ ]:
def normalize_designer_name(value):
    """Normalize spelling format without merging different identities."""
    text = unicodedata.normalize("NFKC", str(value or ""))
    return re.sub(r"\s+", " ", text).strip()


def is_unknown_designer(name):
    """Identify explicit unknown-designer labels."""
    return bool(
        re.search(
            r"\bunknown\b|未知|佚名",
            name,
            flags=re.IGNORECASE
        )
    )


def extract_designer_features(row):
    """Extract identified designers while preserving uncertainty."""
    entries = row.get("designers", [])

    if not isinstance(entries, list):
        raise ValueError("The designers field must contain a list.")

    all_names = list(dict.fromkeys(
        normalize_designer_name(entry.get("name", ""))
        for entry in entries
        if normalize_designer_name(entry.get("name", ""))
    ))

    identified_names = [
        name for name in all_names
        if not is_unknown_designer(name)
    ]

    explicit_unknown = any(
        is_unknown_designer(name)
        for name in all_names
    )

    raw_text = normalize_designer_name(
        row.get("designers_raw", "")
    )

    if not all_names:
        if not raw_text or is_unknown_designer(raw_text):
            status = "unknown"
        else:
            # A nonempty raw field without parsed names needs inspection.
            status = "unparsed"
    elif not identified_names:
        status = "unknown"
    elif explicit_unknown:
        status = "partially_identified"
    else:
        status = "identified"

    if status == "identified":
        multi_designer = int(len(identified_names) > 1)
    elif len(identified_names) > 1:
        # At least two identified entities are already present.
        multi_designer = 1
    else:
        multi_designer = np.nan

    return {
        "designer_names_identified": identified_names,
        "designer_identified_count": len(identified_names),
        "designer_has_identified": int(bool(identified_names)),
        "designer_identity_status": status,
        "multi_designer": multi_designer
    }

In [ ]:
designer_values = pd.DataFrame([
    extract_designer_features(row)
    for _, row in eligible_posters.iterrows()
])

designer_features = pd.concat(
    [
        eligible_posters[
            ["poster_id", "designers_raw", "source_url"]
        ].reset_index(drop=True),
        designer_values
    ],
    axis=1
)

assert designer_features["poster_id"].is_unique
assert len(designer_features) == len(eligible_posters)

print("Designer identity status:")
print(
    designer_features["designer_identity_status"]
    .value_counts()
)

print("\nMultiple-designer indicator:")
print(
    designer_features["multi_designer"]
    .value_counts(dropna=False)
)

display(
    designer_features[
        [
            "poster_id",
            "designers_raw",
            "designer_names_identified",
            "designer_identity_status",
            "multi_designer"
        ]
    ].head(15)
)

In [ ]:
identified = designer_features.loc[
    designer_features["designer_has_identified"] == 1,
    [
        "poster_id",
        "source_url",
        "designer_names_identified"
    ]
].copy()

designer_mentions = (
    identified
    .explode("designer_names_identified")
    .rename(columns={
        "designer_names_identified": "designer_name"
    })
    .drop_duplicates(["poster_id", "designer_name"])
)

designer_entity_review = (
    designer_mentions.groupby("designer_name")
    .agg(
        poster_count=("poster_id", "nunique"),
        example_poster_id=("poster_id", "first"),
        example_source_url=("source_url", "first")
    )
    .reset_index()
    .sort_values(
        ["poster_count", "designer_name"],
        ascending=[False, True]
    )
)

designer_entity_review["entity_type"] = ""
designer_entity_review["reviewed"] = 0
designer_entity_review["evidence"] = ""
designer_entity_review["coder"] = ""

print("Identified designer entities:",
      len(designer_entity_review))

display(designer_entity_review.head(30))

In [ ]:
designer_features_csv = designer_features.copy()

designer_features_csv["designer_names_identified"] = (
    designer_features_csv["designer_names_identified"]
    .apply(lambda names: json.dumps(names, ensure_ascii=False))
)

designer_features_csv.to_csv(
    processed_folder / "poster_designer_features_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

review_path = (
    review_folder / "designer_entities_to_review.csv"
)

if not review_path.exists():
    designer_entity_review.to_csv(
        review_path,
        index=False,
        encoding="utf-8-sig"
    )
    print("Designer entity review file created.")
else:
    print("Existing manual review file preserved.")

In [ ]:
organization_pattern = re.compile(
    r"出版社|出版局|博物馆|学院|大学|学校|委员会|宣传部|"
    r"管理局|地震局|通讯社|新华社|画报社|报社|"
    r"\b(?:press|museum|academy|university|school|"
    r"committee|department|administration|association|"
    r"agency|pictorial)\b",
    flags=re.IGNORECASE
)

collective_pattern = re.compile(
    r"集体创作|创作组|创作小组|"
    r"\b(?:collective creation|creation group|"
    r"creative team)\b",
    flags=re.IGNORECASE
)


def suggest_entity_type(name):
    """Suggest categories for review; do not confirm identities."""
    if collective_pattern.search(name):
        return pd.Series({
            "suggested_entity_type": "collective",
            "suggestion_basis": "explicit_collective_wording"
        })

    if organization_pattern.search(name):
        return pd.Series({
            "suggested_entity_type": "organization",
            "suggestion_basis": "explicit_organization_wording"
        })

    return pd.Series({
        "suggested_entity_type": "needs_review",
        "suggestion_basis": "no_decisive_rule"
    })


suggestions = designer_entity_review[
    "designer_name"
].apply(suggest_entity_type)

designer_entity_review = (
    designer_entity_review
    .drop(columns=suggestions.columns, errors="ignore")
    .join(suggestions)
)

print("Suggested categories:")
print(
    designer_entity_review["suggested_entity_type"]
    .value_counts()
)

display(
    designer_entity_review[
        [
            "designer_name",
            "poster_count",
            "suggested_entity_type",
            "suggestion_basis",
            "entity_type",
            "reviewed"
        ]
    ].head(30)
)

In [ ]:
suggestion_path = (
    review_folder / "designer_entity_suggestions.csv"
)

designer_entity_review.to_csv(
    suggestion_path,
    index=False,
    encoding="utf-8-sig"
)

print("Suggestions saved to:", suggestion_path.resolve())

In [ ]:
def extract_designer_rule_features(row):
    """Create automated name-based indicators, not verified identities."""
    names = row["designer_names_identified"]
    identity_status = row["designer_identity_status"]

    organization_hits = [
        name for name in names
        if organization_pattern.search(name)
    ]

    collective_hits = [
        name for name in names
        if collective_pattern.search(name)
    ]

    # Positive matches are observable even when other designers are unknown.
    # A zero is assigned only when the designer list is fully identified.
    def presence_indicator(hits):
        if hits:
            return 1

        if identity_status == "identified":
            return 0

        return np.nan

    return {
        "designer_org_name_marker": presence_indicator(
            organization_hits
        ),
        "designer_collective_name_marker": presence_indicator(
            collective_hits
        ),
        "designer_org_marker_matches": organization_hits,
        "designer_collective_marker_matches": collective_hits
    }


rule_values = pd.DataFrame([
    extract_designer_rule_features(row)
    for _, row in designer_features.iterrows()
])

designer_features_auto = pd.concat(
    [
        designer_features.reset_index(drop=True),
        rule_values
    ],
    axis=1
)

assert designer_features_auto["poster_id"].is_unique

print("Organization-name marker:")
print(
    designer_features_auto["designer_org_name_marker"]
    .value_counts(dropna=False)
)

print("\nCollective-name marker:")
print(
    designer_features_auto["designer_collective_name_marker"]
    .value_counts(dropna=False)
)

In [ ]:
designer_auto_csv = designer_features_auto.copy()

list_columns = [
    "designer_names_identified",
    "designer_org_marker_matches",
    "designer_collective_marker_matches"
]

for column in list_columns:
    designer_auto_csv[column] = designer_auto_csv[column].apply(
        lambda values: json.dumps(values, ensure_ascii=False)
    )

designer_auto_csv.to_csv(
    processed_folder / "poster_designer_features_auto_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Automated designer features saved.")

In [ ]:
def extract_publisher_features(value):
    """Extract publisher-name indicators without inferring affiliation."""
    raw = "" if pd.isna(value) else str(value)

    normalized = unicodedata.normalize("NFKC", raw)
    normalized = re.sub(r"\s+", " ", normalized).strip()

    if not normalized:
        status = "missing"
    elif re.search(
        r"\bunknown\b|未知|不详|佚名",
        normalized,
        flags=re.IGNORECASE
    ):
        status = "unknown"
    else:
        status = "named"

    patterns = {
        "publisher_press_name_marker": (
            r"出版社|出版局|出版公司|"
            r"\b(?:press|publishing|publisher)\b"
        ),
        "publisher_party_name_marker": (
            r"共产党|国民党|中共|"
            r"\b(?:CCP|CPC|Kuomintang|Communist Party)\b"
        ),
        "publisher_military_name_marker": (
            r"解放军|军队|军区|政治部|"
            r"\b(?:People['’]s Liberation Army|PLA|"
            r"army|military|navy|air force)\b"
        ),
        "publisher_education_name_marker": (
            r"大学|学院|学校|"
            r"\b(?:university|academy|school|college)\b"
        )
    }

    result = {
        "publisher_name_normalized": normalized,
        "publisher_identity_status": status,
        "publisher_has_name": int(status == "named")
    }

    for column, pattern in patterns.items():
        result[column] = (
            int(bool(re.search(
                pattern,
                normalized,
                flags=re.IGNORECASE
            )))
            if status == "named"
            else np.nan
        )

    return result

In [ ]:
publisher_values = pd.DataFrame([
    extract_publisher_features(value)
    for value in eligible_posters["publisher_raw"]
])

publisher_features = pd.concat(
    [
        eligible_posters[
            ["poster_id", "publisher_raw"]
        ].reset_index(drop=True),
        publisher_values
    ],
    axis=1
)

assert publisher_features["poster_id"].is_unique
assert len(publisher_features) == len(eligible_posters)

print("Publisher identity status:")
print(
    publisher_features["publisher_identity_status"]
    .value_counts()
)

marker_columns = [
    "publisher_press_name_marker",
    "publisher_party_name_marker",
    "publisher_military_name_marker",
    "publisher_education_name_marker"
]

for column in marker_columns:
    print(f"\n{column}:")
    print(
        publisher_features[column]
        .value_counts(dropna=False)
    )

In [ ]:
publisher_features.to_csv(
    processed_folder / "poster_publisher_features_auto_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Automated publisher features saved.")

In [ ]:
designer_columns = [
    "poster_id",
    "designer_has_identified",
    "designer_identified_count",
    "designer_identity_status",
    "multi_designer",
    "designer_org_name_marker",
    "designer_collective_name_marker"
]

publisher_columns = [
    "poster_id",
    "publisher_has_name",
    "publisher_identity_status",
    *marker_columns
]

group2_features = (
    eligible_posters[["poster_id"]]
    .merge(
        title_features_selected,
        on="poster_id",
        how="left",
        validate="one_to_one",
        indicator="title_join"
    )
    .merge(
        designer_features_auto[designer_columns],
        on="poster_id",
        how="left",
        validate="one_to_one",
        indicator="designer_join"
    )
    .merge(
        publisher_features[publisher_columns],
        on="poster_id",
        how="left",
        validate="one_to_one",
        indicator="publisher_join"
    )
)

join_columns = [
    "title_join",
    "designer_join",
    "publisher_join"
]

for column in join_columns:
    assert group2_features[column].eq("both").all(), (
        f"Unmatched records in {column}"
    )

group2_features = group2_features.drop(columns=join_columns)

assert len(group2_features) == len(eligible_posters)
assert group2_features["poster_id"].is_unique

group2_features.to_csv(
    processed_folder / "poster_group2_features_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Group 2 feature rows:", len(group2_features))
display(group2_features.head())

In [ ]:
def extract_description_features(value):
    """Measure website-description availability and length."""
    raw = "" if pd.isna(value) else str(value)

    normalized = unicodedata.normalize("NFKC", raw)
    normalized = re.sub(r"\s+", " ", normalized).strip()

    words = re.findall(
        r"[A-Za-z]+(?:['’-][A-Za-z]+)*",
        normalized
    )

    return {
        "description_normalized": normalized,
        "description_present": int(bool(normalized)),
        "description_char_count": (
            len(normalized) if normalized else np.nan
        ),
        "description_english_word_count": (
            len(words) if normalized else np.nan
        )
    }


description_values = pd.DataFrame([
    extract_description_features(value)
    for value in eligible_posters["notes_raw"]
])

In [ ]:
def extract_site_names(entries):
    """Extract unique website labels without inventing categories."""
    if not isinstance(entries, list):
        raise ValueError("Expected a list of website labels.")

    names = [
        re.sub(
            r"\s+",
            " ",
            unicodedata.normalize(
                "NFKC",
                str(entry.get("name", ""))
            )
        ).strip()
        for entry in entries
    ]

    return list(dict.fromkeys(
        name for name in names if name
    ))


site_features = eligible_posters[
    ["poster_id"]
].reset_index(drop=True).copy()

site_features["theme_names"] = [
    extract_site_names(entries)
    for entries in eligible_posters["themes_site"]
]

site_features["tag_names"] = [
    extract_site_names(entries)
    for entries in eligible_posters["tags_site"]
]

site_features["theme_present"] = (
    site_features["theme_names"].apply(bool).astype(int)
)

site_features["tag_present"] = (
    site_features["tag_names"].apply(bool).astype(int)
)

site_features["theme_count"] = (
    site_features["theme_names"].apply(len)
)

site_features["tag_count"] = (
    site_features["tag_names"].apply(len)
)

In [ ]:
theme_frequency = (
    site_features[["poster_id", "theme_names"]]
    .explode("theme_names")
    .dropna(subset=["theme_names"])
    .groupby("theme_names")["poster_id"]
    .nunique()
    .sort_values(ascending=False)
    .rename("poster_count")
    .reset_index()
)

tag_frequency = (
    site_features[["poster_id", "tag_names"]]
    .explode("tag_names")
    .dropna(subset=["tag_names"])
    .groupby("tag_names")["poster_id"]
    .nunique()
    .sort_values(ascending=False)
    .rename("poster_count")
    .reset_index()
)

print("Most frequent website themes:")
display(theme_frequency.head(30))

print("Most frequent website tags:")
display(tag_frequency.head(30))

In [ ]:
group3_features = pd.concat(
    [site_features, description_values],
    axis=1
)

assert group3_features["poster_id"].is_unique
assert len(group3_features) == len(eligible_posters)

group3_csv = group3_features.copy()

for column in ["theme_names", "tag_names"]:
    group3_csv[column] = group3_csv[column].apply(
        lambda values: json.dumps(values, ensure_ascii=False)
    )

group3_csv.to_csv(
    processed_folder / "poster_group3_archive_features_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

theme_frequency.to_csv(
    processed_folder / "site_theme_inventory.csv",
    index=False,
    encoding="utf-8-sig"
)

tag_frequency.to_csv(
    processed_folder / "site_tag_inventory.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Archive feature group saved.")

In [ ]:
def normalize_site_label(value):
    """Normalize a complete website label for exact matching."""
    text = unicodedata.normalize("NFKC", str(value))
    return re.sub(r"\s+", " ", text).strip().casefold()


THEME_RULES = {
    "archive_theme_military": {
        "People's Liberation Army",
        "People's Militia"
    }
}

TAG_RULES = {
    "archive_tag_military_personnel": {
        "soldier",
        "People's Liberation Army",
        "解放军"
    },
    "archive_tag_rifle": {"rifle"},
    "archive_tag_worker": {"worker"},
    "archive_tag_peasant": {"peasant"},
    "archive_tag_female": {"woman", "girl"},
    "archive_tag_child": {"child"},
    "archive_tag_agriculture": {"agriculture"},
    "archive_tag_mao": {"Mao Zedong", "毛泽东"},
    "archive_tag_red_flag": {"red flag"}
}

THEME_RULES = {
    column: {normalize_site_label(label) for label in labels}
    for column, labels in THEME_RULES.items()
}

TAG_RULES = {
    column: {normalize_site_label(label) for label in labels}
    for column, labels in TAG_RULES.items()
}

In [ ]:
def match_site_labels(names, accepted_labels):
    """Return presence within available website metadata."""
    if not names:
        return np.nan

    normalized_names = {
        normalize_site_label(name)
        for name in names
    }

    return int(bool(normalized_names & accepted_labels))


for column, accepted_labels in THEME_RULES.items():
    group3_features[column] = (
        group3_features["theme_names"]
        .apply(lambda names: match_site_labels(
            names, accepted_labels
        ))
    )

for column, accepted_labels in TAG_RULES.items():
    group3_features[column] = (
        group3_features["tag_names"]
        .apply(lambda names: match_site_labels(
            names, accepted_labels
        ))
    )

archive_indicator_columns = [
    *THEME_RULES.keys(),
    *TAG_RULES.keys()
]

display(
    group3_features[archive_indicator_columns]
    .agg(["count", "mean"])
    .T
    .rename(columns={
        "count": "valid_poster_count",
        "mean": "matched_share"
    })
)

In [ ]:
archive_rules = {
    "version": "archive_rules_v1",
    "matching": "normalized exact label matching",
    "scope": "retrospective website metadata",
    "theme_rules": {
        key: sorted(values)
        for key, values in THEME_RULES.items()
    },
    "tag_rules": {
        key: sorted(values)
        for key, values in TAG_RULES.items()
    }
}

(processed_folder / "archive_label_rules_v1.json").write_text(
    json.dumps(archive_rules, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

group3_csv = group3_features.copy()

for column in ["theme_names", "tag_names"]:
    group3_csv[column] = group3_csv[column].apply(
        lambda values: json.dumps(values, ensure_ascii=False)
    )

group3_csv.to_csv(
    processed_folder / "poster_group3_archive_features_v2.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Archive indicators and matching rules saved.")

In [ ]:
archive_selected_columns = [
    "poster_id",
    "description_present",
    "description_char_count",
    "description_english_word_count",
    "theme_present",
    "tag_present",
    "theme_count",
    "tag_count",
    *archive_indicator_columns
]

archive_selected = group3_features[
    archive_selected_columns
].copy()

# Prefix the remaining archive fields consistently.
archive_selected = archive_selected.rename(columns={
    column: f"archive_{column}"
    for column in archive_selected.columns
    if column != "poster_id"
    and not column.startswith("archive_")
})

poster_x = (
    visual_features
    .merge(
        group2_features,
        on="poster_id",
        how="left",
        validate="one_to_one",
        indicator="group2_join"
    )
    .merge(
        archive_selected,
        on="poster_id",
        how="left",
        validate="one_to_one",
        indicator="group3_join"
    )
)

assert poster_x["group2_join"].eq("both").all()
assert poster_x["group3_join"].eq("both").all()

poster_x = poster_x.drop(
    columns=["group2_join", "group3_join"]
)

assert poster_x["poster_id"].is_unique
assert len(poster_x) == len(eligible_posters)

poster_x.to_csv(
    processed_folder / "poster_x_features_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Combined poster-level X shape:", poster_x.shape)

In [ ]:
# Group 1: visual and physical-size features.
group1_columns = visual_columns.copy()

# Group 2: basic title, designer, and publisher indicators.
group2_basic_columns = [
    "title_has_chinese",
    "title_han_count",
    "designer_has_identified",
    "multi_designer",
    "publisher_has_name"
]

# Optional name-rule indicators; not verified affiliations.
group2_rule_columns = [
    "designer_org_name_marker",
    "designer_collective_name_marker",
    "publisher_press_name_marker",
    "publisher_party_name_marker",
    "publisher_military_name_marker",
    "publisher_education_name_marker"
]

# Group 3: retrospective website metadata.
group3_columns = [
    "archive_description_present",
    "archive_description_char_count",
    "archive_description_english_word_count",
    "archive_theme_present",
    "archive_tag_present",
    "archive_theme_count",
    "archive_tag_count",
    *archive_indicator_columns
]

feature_columns = list(dict.fromkeys(
    group1_columns
    + group2_basic_columns
    + group2_rule_columns
    + group3_columns
))

assert all(
    column in poster_x.columns
    for column in feature_columns
), "Some selected feature columns are missing."

assert all(
    pd.api.types.is_numeric_dtype(poster_x[column])
    for column in feature_columns
), "Selected features must be numeric."

print("Selected poster-level numeric features:", len(feature_columns))

In [ ]:
annual_rows = []

for year, group in poster_x.groupby("poster_year"):
    row = {
        "poster_year": int(year),
        "target_year": int(year) + 1,
        "n_posters": len(group)
    }

    for column in feature_columns:
        values = group[column]

        row[f"{column}_mean"] = values.mean()
        row[f"{column}_n_valid"] = int(values.notna().sum())
        row[f"{column}_missing_ratio"] = float(
            values.isna().mean()
        )

    annual_rows.append(row)

annual_x = (
    pd.DataFrame(annual_rows)
    .sort_values("poster_year")
    .reset_index(drop=True)
)

assert annual_x["poster_year"].is_unique
assert len(annual_x) == 65
assert annual_x["n_posters"].sum() == len(poster_x)

print("Annual X shape:", annual_x.shape)

display(
    annual_x[
        [
            "poster_year",
            "target_year",
            "n_posters",
            "red_ratio_mean",
            "multi_designer_mean",
            "archive_tag_military_personnel_mean"
        ]
    ].head(10)
)

In [ ]:
def annual_mean_columns(columns):
    return [f"{column}_mean" for column in columns]


feature_sets = {
    "visual_baseline": annual_mean_columns(
        group1_columns
    ),
    "visual_plus_basic_metadata": annual_mean_columns(
        group1_columns + group2_basic_columns
    ),
    "visual_plus_metadata_rules": annual_mean_columns(
        group1_columns
        + group2_basic_columns
        + group2_rule_columns
    ),
    "retrospective_archive_extension": annual_mean_columns(
        feature_columns
    )
}

annual_x.to_csv(
    processed_folder / "annual_x_features_v1.csv",
    index=False,
    encoding="utf-8-sig"
)

(processed_folder / "feature_sets_v1.json").write_text(
    json.dumps(feature_sets, indent=2),
    encoding="utf-8"
)

print("Annual X and feature-set definitions saved.")

In [ ]:
# Fixed chronological split for one-year-ahead prediction.
split_definition = {
    "version": "chronological_split_v1",
    "prediction_horizon_years": 1,
    "development_poster_years": list(range(1949, 2000)),
    "embargo_poster_years": [2000],
    "test_poster_years": list(range(2001, 2014))
}

results_folder = Path("results")
results_folder.mkdir(parents=True, exist_ok=True)

split_path = results_folder / "split_manifest.json"

# Preserve an existing split instead of silently changing it.
if split_path.exists():
    existing_split = json.loads(
        split_path.read_text(encoding="utf-8")
    )

    if existing_split != split_definition:
        raise ValueError(
            "An existing split differs. Review it before proceeding."
        )
else:
    split_path.write_text(
        json.dumps(split_definition, indent=2),
        encoding="utf-8"
    )

development_years = split_definition["development_poster_years"]

x_dev = poster_x.loc[
    poster_x["poster_year"].isin(development_years)
].copy()

annual_x_dev = annual_x.loc[
    annual_x["poster_year"].isin(development_years)
].copy()

assert not set(x_dev["poster_year"]) & set(
    split_definition["test_poster_years"]
)

print("Development posters:", len(x_dev))
print("Development years:", annual_x_dev["poster_year"].nunique())
print("Test years reserved; no test plots will be generated.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

figure_folder = Path("figures/x_eda")
figure_folder.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False
})


def save_x_figure(fig, filename, caption):
    """Save a figure and its technical caption."""
    fig.tight_layout()

    fig.savefig(
        figure_folder / f"{filename}.png",
        dpi=200,
        bbox_inches="tight"
    )

    fig.savefig(
        figure_folder / f"{filename}.svg",
        bbox_inches="tight"
    )

    (figure_folder / f"{filename}_caption.txt").write_text(
        caption,
        encoding="utf-8"
    )

    plt.show()
    plt.close(fig)

In [ ]:
coverage = (
    x_dev.groupby("poster_year")
    .size()
    .reindex(development_years, fill_value=0)
)

fig, ax = plt.subplots(figsize=(11, 4))

ax.bar(
    coverage.index,
    coverage.values,
    color="#526D82"
)

ax.set_title("Poster coverage differs across development years")
ax.set_xlabel("Poster year")
ax.set_ylabel("Retained catalogue records")

save_x_figure(
    fig,
    "01_poster_coverage",
    (
        "Development years, 1949–1999. Counts refer to retained "
        "catalogue records with eligible dates, not historical "
        "poster production, circulation, or exposure."
    )
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(
    x_dev["red_ratio"].dropna() * 100,
    bins=np.linspace(0, 100, 26),
    color="#B54A45",
    edgecolor="white"
)

axes[0].set_title("Red coverage varies across posters")
axes[0].set_xlabel("Red pixels (%)")
axes[0].set_ylabel("Catalogue records")

axes[1].hist(
    x_dev["lightness_mean"].dropna(),
    bins=np.linspace(0, 100, 26),
    color="#526D82",
    edgecolor="white"
)

axes[1].set_title("Scanned lightness varies across posters")
axes[1].set_xlabel("Mean CIELAB L* (0–100)")
axes[1].set_ylabel("Catalogue records")

save_x_figure(
    fig,
    "02_color_distributions",
    (
        "Development posters only. Red coverage uses the fixed "
        "HSV rule: hue within 15 degrees of red, S >= 0.40 and "
        "V >= 0.15. Lightness is calculated in CIELAB under D65. "
        "The measures describe full digital images and may be "
        "affected by scanning and preservation."
    )
)

In [ ]:
red_by_year = (
    x_dev.groupby("poster_year")["red_ratio"]
    .agg(
        median="median",
        q25=lambda values: values.quantile(0.25),
        q75=lambda values: values.quantile(0.75),
        n_valid="count"
    )
    .reindex(development_years)
)

years = red_by_year.index.to_numpy(dtype=int)

fig, ax = plt.subplots(figsize=(11, 4))

ax.fill_between(
    years,
    red_by_year["q25"].to_numpy(dtype=float) * 100,
    red_by_year["q75"].to_numpy(dtype=float) * 100,
    color="#B54A45",
    alpha=0.2,
    label="Interquartile range"
)

ax.plot(
    years,
    red_by_year["median"].to_numpy(dtype=float) * 100,
    color="#B54A45",
    linewidth=1.8,
    label="Median"
)

ax.set_title("Annual red coverage includes substantial within-year variation")
ax.set_xlabel("Poster year")
ax.set_ylabel("Red pixels (%)")
ax.legend(frameon=False)

save_x_figure(
    fig,
    "03_annual_red_coverage",
    (
        "Development years only. The line shows the median "
        "poster-level red coverage; shading shows the 25th–75th "
        "percentiles, not a confidence interval. Each catalogue "
        "record receives equal weight within its year. Annual "
        "sample counts should be read alongside Figure 1."
    )
)

red_by_year.to_csv(
    figure_folder / "03_annual_red_coverage_data.csv",
    encoding="utf-8-sig"
)

In [ ]:
marker_labels = {
    "designer_org_name_marker": "Designer: organization wording",
    "designer_collective_name_marker": "Designer: collective wording",
    "publisher_press_name_marker": "Publisher: press wording",
    "publisher_party_name_marker": "Publisher: party wording",
    "publisher_military_name_marker": "Publisher: military wording",
    "publisher_education_name_marker": "Publisher: education wording"
}

marker_summary = pd.DataFrame([
    {
        "feature": column,
        "label": label,
        "matched_share": x_dev[column].mean(),
        "n_valid": int(x_dev[column].notna().sum()),
        "n_missing": int(x_dev[column].isna().sum())
    }
    for column, label in marker_labels.items()
]).sort_values("matched_share")

fig, ax = plt.subplots(figsize=(10, 5))

y_positions = np.arange(len(marker_summary))

ax.barh(
    y_positions,
    marker_summary["matched_share"] * 100,
    color="#526D82"
)

ax.set_yticks(
    y_positions,
    [
        f"{row.label} (n={row.n_valid})"
        for row in marker_summary.itertuples()
    ]
)

ax.set_xlim(0, 100)
ax.set_xlabel("Matched records among nonmissing records (%)")
ax.set_title("Name-based markers capture different organizational signals")

save_x_figure(
    fig,
    "04_organization_name_markers",
    (
        "Development posters only. Each proportion uses nonmissing "
        "records for that indicator. Markers are automated matches "
        "to name wording, not verified organizational identities or "
        "party/military affiliations. Indicators are nonexclusive."
    )
)

marker_summary.to_csv(
    figure_folder / "04_organization_name_markers_data.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
availability_rows = [
    {
        "feature": "Chinese title unavailable",
        "unavailable_share": 1 - x_dev["title_has_chinese"].mean()
    },
    {
        "feature": "No identified designer",
        "unavailable_share": 1 - x_dev["designer_has_identified"].mean()
    },
    {
        "feature": "Publisher name unavailable",
        "unavailable_share": 1 - x_dev["publisher_has_name"].mean()
    },
    {
        "feature": "Physical size unparsed or missing",
        "unavailable_share": x_dev["size_area_cm2"].isna().mean()
    },
    {
        "feature": "Website description unavailable",
        "unavailable_share": 1 - x_dev["archive_description_present"].mean()
    },
    {
        "feature": "Website themes unavailable",
        "unavailable_share": 1 - x_dev["archive_theme_present"].mean()
    },
    {
        "feature": "Website tags unavailable",
        "unavailable_share": 1 - x_dev["archive_tag_present"].mean()
    }
]

availability_summary = (
    pd.DataFrame(availability_rows)
    .sort_values("unavailable_share")
)

fig, ax = plt.subplots(figsize=(10, 5))

ax.barh(
    availability_summary["feature"],
    availability_summary["unavailable_share"] * 100,
    color="#8C6D62"
)

ax.set_xlim(0, 100)
ax.set_xlabel("Development records (%)")
ax.set_title("Metadata availability differs across feature sources")

save_x_figure(
    fig,
    "05_feature_availability",
    (
        "Development posters only. Unavailability follows the "
        "preprocessing definitions: absent Chinese text, no identified "
        "designer, unknown or missing publisher, unsupported physical "
        "size, or absent website metadata. It does not imply absence "
        "of the underlying historical content."
    )
)

availability_summary.to_csv(
    figure_folder / "05_feature_availability_data.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
archive_labels = {
    "archive_theme_military": "Military theme",
    "archive_tag_military_personnel": "Military personnel",
    "archive_tag_rifle": "Rifle",
    "archive_tag_worker": "Worker",
    "archive_tag_peasant": "Peasant",
    "archive_tag_female": "Woman or girl",
    "archive_tag_child": "Child",
    "archive_tag_agriculture": "Agriculture",
    "archive_tag_mao": "Mao Zedong",
    "archive_tag_red_flag": "Red flag",
}

archive_summary = pd.DataFrame([
    {
        "feature": column,
        "label": label,
        "matched_percent": 100 * x_dev[column].mean(),
        "n_valid": int(x_dev[column].notna().sum()),
    }
    for column, label in archive_labels.items()
]).sort_values("matched_percent", ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.barh(
    archive_summary["label"],
    archive_summary["matched_percent"],
    color="#526D82",
)

ax.bar_label(bars, fmt="%.1f%%", padding=4)
ax.set_xlim(0, 100)
ax.set_xlabel("Matched records among available metadata (%)")
ax.set_title("Archive labels describe varied poster subjects")

save_x_figure(
    fig,
    "06_archive_subject_markers",
    "Development-period posters only. Each indicator uses available "
    "theme or tag metadata as its denominator. Categories can overlap. "
    "Labels are retrospective website annotations.",
)

archive_summary.to_csv(
    figure_folder / "06_archive_subject_markers_data.csv",
    index=False,
    encoding="utf-8-sig",
)

display(archive_summary)

In [ ]:
area = pd.to_numeric(
    x_dev["size_area_cm2"], errors="coerce"
).dropna()
area = area[area > 0]

aspect = pd.to_numeric(
    x_dev["size_aspect_ratio"], errors="coerce"
).dropna()
aspect = aspect[aspect >= 1]

assert len(area) > 0, "No valid physical areas available."
assert len(aspect) > 0, "No valid aspect ratios available."

fig, axes = plt.subplots(figsize=(13, 4.5), ncols=2)

area_bins = np.geomspace(
    area.min() * 0.95,
    area.max() * 1.05,
    26,
)

axes[0].hist(
    area,
    bins=area_bins,
    color="#526D82",
    edgecolor="white",
)
axes[0].set_xscale("log")
axes[0].set_xlabel("Physical area (cm², logarithmic scale)")
axes[0].set_ylabel("Catalogue records")
axes[0].set_title("Physical poster areas span a wide range")

axes[1].hist(
    aspect,
    bins=25,
    color="#8C6D62",
    edgecolor="white",
)
axes[1].set_xlabel("Long side / short side")
axes[1].set_ylabel("Catalogue records")
axes[1].set_title("Poster formats cluster around similar proportions")

save_x_figure(
    fig,
    "07_physical_size_distributions",
    "Development-period posters with successfully parsed dimensions. "
    "Area is displayed on a logarithmic axis. Aspect ratio is the "
    "longer side divided by the shorter side; orientation is not measured.",
)

In [ ]:
annual_feature_labels = {
    "red_ratio_mean": "Red coverage",
    "saturation_mean_mean": "Saturation",
    "lightness_mean_mean": "Lightness",
    "lightness_std_mean": "Lightness variation",
    "size_area_cm2_mean": "Physical area",
    "size_aspect_ratio_mean": "Aspect ratio",
    "title_han_count_mean": "Chinese title length",
    "multi_designer_mean": "Multiple designer entities",
    "designer_org_name_marker_mean": "Designer organization wording",
    "publisher_military_name_marker_mean": "Publisher military wording",
}

selected_columns = list(annual_feature_labels)

missing_columns = [
    column for column in selected_columns
    if column not in annual_x_dev.columns
]
assert not missing_columns, f"Missing columns: {missing_columns}"

# Exclude constant features whose correlations are undefined.
annual_values = annual_x_dev[selected_columns].apply(
    pd.to_numeric, errors="coerce"
)

variable_columns = annual_values.columns[
    annual_values.nunique(dropna=True) > 1
]

# Require at least 20 paired years for each correlation.
correlation = annual_values[variable_columns].corr(
    method="spearman",
    min_periods=20,
)

labels = [
    annual_feature_labels[column]
    for column in correlation.columns
]

fig, ax = plt.subplots(figsize=(11, 9))

heatmap = ax.imshow(
    np.ma.masked_invalid(correlation.to_numpy()),
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
)

ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_yticklabels(labels)

for row in range(len(labels)):
    for column in range(len(labels)):
        value = correlation.iloc[row, column]
        if pd.notna(value):
            ax.text(
                column,
                row,
                f"{value:.2f}",
                ha="center",
                va="center",
                fontsize=8,
                color="white" if abs(value) > 0.65 else "black",
            )

fig.colorbar(
    heatmap,
    ax=ax,
    shrink=0.8,
    label="Spearman correlation",
)

ax.set_title("Annual poster features share overlapping information")

save_x_figure(
    fig,
    "08_annual_x_correlations",
    "Development years only. Spearman correlations between annual "
    "feature means, using pairwise available years with at least "
    "20 paired observations. Constant features are excluded. "
    "Correlations do not establish causation.",
)

correlation.to_csv(
    figure_folder / "08_annual_x_correlations_data.csv",
    encoding="utf-8-sig",
)